# PS05: Lunar Lander Estimation and Control

> Self-contained Python notebook with four progressive sections:
1. Simple estimation tuning + test
2. Simple controller tuning + test
3. Simple estimation + controller
4. Complex landing (high diagonal velocity near tower)

> This notebook ports the core logic from the MATLAB PS06 helper files into Python, with conservative default tuning values that should run without further adjustments.

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.linalg import solve_continuous_are
from scipy.signal import place_poles

np.set_printoptions(precision=4, suppress=True)

## Shared Model and Utility Functions

> The functions below replicate the behavior of the MATLAB helpers: nonlinear true dynamics, observer update, LQR feedback law, geometry for drawing, and landing checks.

In [22]:
def compute_lander_shape(x_state: np.ndarray, dims: dict, phi: float):
    """Compute polygon coordinates for the lander body and nozzle line."""
    x_pos, y_pos, theta = x_state[0], x_state[2], x_state[4]

    w = dims["lander_width"]
    h = dims["lander_height"]
    lander_shape = np.array([
        [-w / 2,  w / 2,  w / 2, -w / 2, -w / 2],
        [-h / 2, -h / 2,  h / 2,  h / 2, -h / 2],
    ])

    # Keep the same sign convention as the MATLAB version.
    R = np.array([[np.cos(theta), np.sin(theta)],
                  [-np.sin(theta), np.cos(theta)]])

    lander_rot = R @ lander_shape
    lander_x = lander_rot[0, :] + x_pos
    lander_y = lander_rot[1, :] + y_pos

    nozzle_base_body = np.array([0.0, -h / 2])
    nozzle_dir_body = np.array([np.sin(phi), -np.cos(phi)])
    nozzle_tip_body = nozzle_base_body + dims["nozzle_length"] * nozzle_dir_body

    nozzle_base_rot = R @ nozzle_base_body
    nozzle_tip_rot = R @ nozzle_tip_body

    nozzle_x = np.array([nozzle_base_rot[0] + x_pos, nozzle_tip_rot[0] + x_pos])
    nozzle_y = np.array([nozzle_base_rot[1] + y_pos, nozzle_tip_rot[1] + y_pos])
    return lander_x, lander_y, nozzle_x, nozzle_y


def update_true_state(x: np.ndarray, u: np.ndarray, dt: float, m: float, I: float, g: float, l1: float):
    """Nonlinear lander dynamics integrated with forward Euler."""
    T, phi = u[0], u[1]
    theta = x[4]

    x_ddot = (T * np.sin(theta + phi)) / m
    y_ddot = (-m * g + T * np.cos(theta + phi)) / m
    theta_ddot = (T * l1 * np.sin(phi)) / I

    x_dot = np.array([x[1], x_ddot, x[3], y_ddot, x[5], theta_ddot])
    return x + dt * x_dot


def update_observer(x_hat: np.ndarray, u: np.ndarray, y_meas: np.ndarray, dt: float, A: np.ndarray, B: np.ndarray, C: np.ndarray, L: np.ndarray, g: float = 1.62):
    """Luenberger observer with gravity compensation in vertical acceleration channel."""
    gravity_term = np.array([0.0, 0.0, 0.0, g, 0.0, 0.0])
    x_hat_dot = A @ x_hat + B @ u + L @ (y_meas - C @ x_hat) - gravity_term
    return x_hat + dt * x_hat_dot


def lqr_controller(x_hat: np.ndarray, x_ref: np.ndarray, u_eq: np.ndarray, K: np.ndarray):
    """LQR tracking law: u = u_eq - K (x_hat - x_ref)."""
    return u_eq - K @ (x_hat - x_ref)


def in_zone(xp: float, yp: float, zone: dict):
    """Return true if point lies in an axis-aligned rectangle zone."""
    return (zone["x"] <= xp <= zone["x"] + zone["width"]) and (zone["y"] <= yp <= zone["y"] + zone["height"])


def check_landing(x: np.ndarray, dims: dict, safe_zone: dict, hazard_zone: dict, vel_threshold: float = 0.3):
    """Determine whether lander has landed and whether landing is safe."""
    px, py, _, _ = compute_lander_shape(x, dims, phi=0.0)
    vel = np.array([x[1], x[3]])

    landed = np.any(py <= 0.0)
    success = False

    for i in range(len(px)):
        if in_zone(float(px[i]), float(py[i]), hazard_zone):
            return True, False, vel

    all_in_safe = np.all([in_zone(float(px[i]), float(py[i]), safe_zone) for i in range(len(px))])
    if all_in_safe:
        landed = True
        success = (abs(vel[0]) < vel_threshold) and (abs(vel[1]) < vel_threshold)

    if landed and (not all_in_safe):
        success = False

    return landed, success, vel


def clamp_input(u: np.ndarray, thrust_limits=(0.0, 25.0), phi_limits=(-0.35, 0.35)):
    """Clamp actuator commands to simple physical limits."""
    T = np.clip(u[0], thrust_limits[0], thrust_limits[1])
    phi = np.clip(u[1], phi_limits[0], phi_limits[1])
    return np.array([T, phi])


def lqr_gain(A: np.ndarray, B: np.ndarray, Q: np.ndarray, R: np.ndarray):
    """Compute continuous-time LQR gain using the Riccati equation."""
    P = solve_continuous_are(A, B, Q, R)
    return np.linalg.solve(R, B.T @ P)


def observer_gain_place(A: np.ndarray, C: np.ndarray, poles):
    """Observer gain by pole placement on transposed dual system."""
    placed = place_poles(A.T, C.T, poles)
    return placed.gain_matrix.T

In [23]:
def run_closed_loop(
    x0: np.ndarray, xhat0: np.ndarray, x_ref: np.ndarray,
    A: np.ndarray, B: np.ndarray, C: np.ndarray,
    K: np.ndarray, L: np.ndarray, u_eq: np.ndarray,
    params: dict, mode: str = "full"
    ):
    """Simulate estimation/control modes: estimation_only, controller_only, full."""
    n_steps = int(params["t_final"] / params["dt"])
    dt = params["dt"]

    x = x0.copy()
    x_hat = xhat0.copy()

    X = np.zeros((n_steps + 1, 6))
    XH = np.zeros((n_steps + 1, 6))
    U = np.zeros((n_steps + 1, 2))
    Y = np.zeros((n_steps + 1, C.shape[0]))
    t = np.linspace(0.0, params["t_final"], n_steps + 1)

    X[0] = x
    XH[0] = x_hat
    Y[0] = C @ x

    landed = False
    success = False
    landing_idx = n_steps

    for k in range(n_steps):
        y_meas = C @ x + params["noise_std"] * np.random.randn(C.shape[0])
        Y[k] = y_meas

        if mode == "estimation_only":
            u = u_eq.copy()
            x_hat = update_observer(x_hat, u, y_meas, dt, A, B, C, L, g=params["g"])
        elif mode == "controller_only":
            u = lqr_controller(x, x_ref, u_eq, K)
            u = clamp_input(u, params["thrust_limits"], params["phi_limits"])
            x_hat = x.copy()
        else:
            u = lqr_controller(x_hat, x_ref, u_eq, K)
            u = clamp_input(u, params["thrust_limits"], params["phi_limits"])
            x_hat = update_observer(x_hat, u, y_meas, dt, A, B, C, L, g=params["g"])

        x = update_true_state(x, u, dt, params["m"], params["I"], params["g"], params["l1"])
        landed_now, success_now, _ = check_landing(
            x, params["dims"], params["safe_zone"], params["hazard_zone"], params["vel_threshold"]
        )
        if landed_now and (not landed):
            landed = True
            success = success_now
            landing_idx = k + 1

        X[k + 1] = x
        XH[k + 1] = x_hat
        U[k + 1] = u

        if landed and params["stop_on_landing"]:
            X = X[: k + 2]
            XH = XH[: k + 2]
            U = U[: k + 2]
            Y = Y[: k + 2]
            t = t[: k + 2]
            break

    Y[-1] = C @ X[-1]
    return {
        "t": t,
        "X": X,
        "XH": XH,
        "U": U,
        "Y": Y,
        "landed": landed,
        "success": success,
        "landing_idx": landing_idx,
    }


def plot_core_results(result: dict, title: str):
    """Compact multi-panel plot for trajectory, states, and control."""
    t = result["t"]
    X = result["X"]
    XH = result["XH"]
    U = result["U"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(X[:, 0], X[:, 2], label="True trajectory")
    axes[0, 0].set_xlabel("x [m]")
    axes[0, 0].set_ylabel("y [m]")
    axes[0, 0].set_title("Position trajectory")
    axes[0, 0].grid(True)
    axes[0, 0].axis("equal")

    axes[0, 1].plot(t, X[:, 0], label="x")
    axes[0, 1].plot(t, X[:, 2], label="y")
    axes[0, 1].plot(t, XH[:, 0], "--", label="x_hat")
    axes[0, 1].plot(t, XH[:, 2], "--", label="y_hat")
    axes[0, 1].set_title("Position and estimates")
    axes[0, 1].set_xlabel("t [s]")
    axes[0, 1].grid(True)
    axes[0, 1].legend()

    axes[1, 0].plot(t, X[:, 1], label="dx")
    axes[1, 0].plot(t, X[:, 3], label="dy")
    axes[1, 0].plot(t, X[:, 4], label="theta")
    axes[1, 0].set_title("Velocity and attitude")
    axes[1, 0].set_xlabel("t [s]")
    axes[1, 0].grid(True)
    axes[1, 0].legend()

    axes[1, 1].plot(t, U[:, 0], label="T [N]")
    axes[1, 1].plot(t, U[:, 1], label="phi [rad]")
    axes[1, 1].set_title("Control inputs")
    axes[1, 1].set_xlabel("t [s]")
    axes[1, 1].grid(True)
    axes[1, 1].legend()

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def plot_landing_scene(result: dict, params: dict, title: str):
    """Plot trajectory with safe/hazard zones and final lander body pose."""
    X = result["X"]
    safe = params["safe_zone"]
    hazard = params["hazard_zone"]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(X[:, 0], X[:, 2], lw=2.0, label="Trajectory")

    safe_rect = plt.Rectangle((safe["x"], safe["y"]), safe["width"], safe["height"],
                              alpha=0.25, color="green", label="Safe zone")
    hazard_rect = plt.Rectangle((hazard["x"], hazard["y"]), hazard["width"], hazard["height"],
                                alpha=0.35, color="black", label="Hazard zone")
    ax.add_patch(safe_rect)
    ax.add_patch(hazard_rect)

    x_final = X[-1]
    lander_x, lander_y, nozzle_x, nozzle_y = compute_lander_shape(x_final, params["dims"], 0.0)
    ax.plot(lander_x, lander_y, color="tab:blue", lw=2.0, label="Final lander")
    ax.plot(nozzle_x, nozzle_y, color="tab:red", lw=2.0, label="Nozzle")

    ax.axhline(0.0, color="gray", ls="--", lw=1.0)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()
    ax.axis("equal")
    plt.show()

## Default Parameters and Linear Model

> These defaults are chosen to be conservative and stable for all four sections.

In [24]:
# Physical and simulation parameters
params = {
    "m": 1.0,
    "I": 0.02,
    "g": 1.62,
    "l1": 0.20,
    "dt": 0.01,
    "t_final": 25.0,
    "noise_std": 0.01,
    "vel_threshold": 0.3,
    "thrust_limits": (0.0, 30.0),
    "phi_limits": (-0.35, 0.35),
    "stop_on_landing": True,
    "dims": {"lander_width": 0.60, "lander_height": 0.35, "nozzle_length": 0.18},
    "safe_zone": {"x": -0.9, "y": 0.0, "width": 1.8, "height": 0.8},
    "hazard_zone": {"x": 1.2, "y": 0.0, "width": 0.45, "height": 2.4},
}

# Store latest metrics for optional recap.
section_metrics = {}

# State: [x, dx, y, dy, theta, dtheta]
x_ref = np.array([0.0, 0.0, 0.35, 0.0, 0.0, 0.0])
u_eq = np.array([params["m"] * params["g"], 0.0])

# Linearized model around hover (theta = 0, phi = 0, T = m g)
g = params["g"]
A = np.array([
    [0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, g, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0],
], dtype=float)

B = np.array([
    [0, 0],
    [0, g],
    [0, 0],
    [1 / params["m"], 0],
    [0, 0],
    [0, params["m"] * params["g"] * params["l1"] / params["I"]],
], dtype=float)

# Measurements: x, y, theta
C = np.array([
    [1, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 0],
], dtype=float)

Q = np.diag([12.0, 2.0, 20.0, 4.0, 8.0, 2.0])
R = np.diag([0.55, 2.2])
K = lqr_gain(A, B, Q, R)

observer_poles = np.array([-3.2, -3.6, -4.0, -4.3, -4.7, -5.0])
L = observer_gain_place(A, C, observer_poles)

print("K shape:", K.shape, "L shape:", L.shape)
print("u_eq:", u_eq)

K shape: (2, 6) L shape: (6, 3)
u_eq: [1.62 0.  ]


## 1) Simple Estimation Tuning + Test

> Goal: tune/verify only the observer while keeping the plant near hover with fixed input.

In [32]:
x0 = np.array([0.8, -0.25, 2.1, -0.45, 0.18, -0.10])
xhat0 = np.array([0.0, 0.0, 1.2, 0.0, 0.0, 0.0])

def run_section1(obs_speed=1.0, noise_std=0.01):
    np.random.seed(5)
    local_params = dict(params)
    local_params["noise_std"] = noise_std

    L_tuned = observer_gain_place(A, C, observer_poles * obs_speed)
    est_result = run_closed_loop(
        x0=x0, xhat0=xhat0, x_ref=x_ref,
        A=A, B=B, C=C, K=K, L=L_tuned, u_eq=u_eq,
        params=local_params, mode="estimation_only"
    )

    plot_core_results(est_result, "Section 1: Estimation Only (Interactive)")
    err = est_result["X"] - est_result["XH"]
    err_norm = np.linalg.norm(err[:, [0, 2, 4]], axis=1)
    converged = bool(err_norm[-1] < 0.25 * err_norm[0])

    section_metrics["sec1"] = {
        "initial_err": float(err_norm[0]),
        "final_err": float(err_norm[-1]),
        "converged": converged,
    }

    print("Initial estimation error norm (x,y,theta):", float(err_norm[0]))
    print("Final estimation error norm (x,y,theta):", float(err_norm[-1]))
    print("Converged (final < 25% of initial):", converged)

sec1_obs_speed = widgets.FloatSlider(value=1.0, min=0.6, max=1.8, step=0.05, description="Obs speed")
sec1_noise = widgets.FloatSlider(value=0.01, min=0.0, max=0.05, step=0.0025, description="Noise std")

sec1_ui = widgets.VBox([sec1_obs_speed, sec1_noise])
sec1_out = widgets.interactive_output(run_section1, {
    "obs_speed": sec1_obs_speed,
    "noise_std": sec1_noise,
})
display(sec1_ui, sec1_out)

Output()

## 2) Simple Controller Tuning + Test

> Goal: test LQR control with full-state feedback (controller-focused test).

In [33]:
x0_ctrl = np.array([-1.8, 0.6, 2.8, -0.8, -0.20, 0.12])
xhat0_ctrl = x0_ctrl.copy()

def run_section2(q_pos_scale=1.0, q_theta_scale=1.0, r_thrust_scale=1.0, r_phi_scale=1.0):
    np.random.seed(7)
    Q_tuned = np.diag([
        12.0 * q_pos_scale, 2.0, 20.0 * q_pos_scale, 4.0,
        8.0 * q_theta_scale, 2.0 * q_theta_scale
    ])
    R_tuned = np.diag([0.55 * r_thrust_scale, 2.2 * r_phi_scale])
    K_tuned = lqr_gain(A, B, Q_tuned, R_tuned)

    ctrl_result = run_closed_loop(
        x0=x0_ctrl, xhat0=xhat0_ctrl, x_ref=x_ref,
        A=A, B=B, C=C, K=K_tuned, L=L, u_eq=u_eq,
        params=params, mode="controller_only"
    )

    plot_core_results(ctrl_result, "Section 2: Controller Only (Interactive)")
    state_err = np.linalg.norm(ctrl_result["X"][-1] - x_ref)
    peak_thrust = float(np.max(ctrl_result["U"][:, 0]))
    peak_phi = float(np.max(np.abs(ctrl_result["U"][:, 1])))
    stable_like = bool(state_err < 0.8)

    section_metrics["sec2"] = {
        "final_state_err": float(state_err),
        "peak_thrust": peak_thrust,
        "peak_phi": peak_phi,
        "stable_like": stable_like,
    }

    print("Final state error norm:", float(state_err))
    print("Peak thrust [N]:", peak_thrust)
    print("Peak |phi| [rad]:", peak_phi)
    print("Controller stable-looking (error < 0.8):", stable_like)

sec2_qpos = widgets.FloatSlider(value=1.0, min=0.4, max=2.5, step=0.05, description="Q pos x/y")
sec2_qtheta = widgets.FloatSlider(value=1.0, min=0.4, max=2.5, step=0.05, description="Q theta")
sec2_rT = widgets.FloatSlider(value=1.0, min=0.4, max=2.5, step=0.05, description="R thrust")
sec2_rphi = widgets.FloatSlider(value=1.0, min=0.4, max=2.5, step=0.05, description="R phi")

sec2_ui = widgets.VBox([sec2_qpos, sec2_qtheta, sec2_rT, sec2_rphi])
sec2_out = widgets.interactive_output(run_section2, {
    "q_pos_scale": sec2_qpos,
    "q_theta_scale": sec2_qtheta,
    "r_thrust_scale": sec2_rT,
    "r_phi_scale": sec2_rphi,
})
display(sec2_ui, sec2_out)

Output()

## 3) Simple Estimation + Controller

> Goal: combine observer and controller. With defaults, this should need little or no tuning.

In [ ]:
x0_full = np.array([-1.4, 0.5, 3.0, -0.65, 0.22, -0.15])
xhat0_full = np.array([-0.4, 0.0, 2.4, 0.0, 0.0, 0.0])

def run_section3(ctrl_aggr=1.0, obs_speed=1.0, noise_std=0.01):
    np.random.seed(11)
    local_params = dict(params)
    local_params["noise_std"] = noise_std

    Q_tuned = Q * ctrl_aggr
    R_tuned = R
    K_tuned = lqr_gain(A, B, Q_tuned, R_tuned)
    L_tuned = observer_gain_place(A, C, observer_poles * obs_speed)

    full_result = run_closed_loop(
        x0=x0_full, xhat0=xhat0_full, x_ref=x_ref,
        A=A, B=B, C=C, K=K_tuned, L=L_tuned, u_eq=u_eq,
        params=local_params, mode="full"
    )

    plot_core_results(full_result, "Section 3: Estimation + Controller (Interactive)")
    full_err = np.linalg.norm(full_result["X"][-1] - x_ref)
    est_err = np.linalg.norm(full_result["X"][-1] - full_result["XH"][-1])
    stable_like = bool(full_err < 1.0)

    section_metrics["sec3"] = {
        "final_tracking_err": float(full_err),
        "final_estimation_err": float(est_err),
        "stable_like": stable_like,
    }

    print("Final tracking error norm:", float(full_err))
    print("Final estimation mismatch norm:", float(est_err))
    print("Combined loop stable-looking (tracking error < 1.0):", stable_like)

sec3_ctrl = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description="Ctrl aggr")
sec3_obs = widgets.FloatSlider(value=1.0, min=0.6, max=1.8, step=0.05, description="Obs speed")
sec3_noise = widgets.FloatSlider(value=0.01, min=0.0, max=0.05, step=0.0025, description="Noise std")

sec3_ui = widgets.VBox([sec3_ctrl, sec3_obs, sec3_noise])
sec3_out = widgets.interactive_output(run_section3, {
    "ctrl_aggr": sec3_ctrl,
    "obs_speed": sec3_obs,
    "noise_std": sec3_noise,
})
display(sec3_ui, sec3_out)

Output()

## 4) Complex Landing: High Diagonal Velocity Near Tower

> Goal: stress test with large diagonal speed and approach close to the hazard tower before touchdown.

In [39]:
base_complex_params = dict(params)
base_complex_params["t_final"] = 50.0
base_complex_params["noise_std"] = 0.015
base_complex_params["safe_zone"] = {"x": -0.5, "y": 0.0, "width": 1.0, "height": 0.85}
base_complex_params["hazard_zone"] = {"x": 1.0, "y": 0.0, "width": 0.40, "height": 2.8}
base_complex_params["stop_on_landing"] = True

x0_complex = np.array([-2.4, 2.1, 4.8, -2.3, 0.26, -0.22])
xhat0_complex = np.array([-1.8, 0.4, 4.2, -0.8, 0.10, 0.00])

def run_section4(target_x=1.10, vel_scale=1.0, ctrl_aggr=1.0, obs_speed=1.0):
    np.random.seed(21)
    complex_params = dict(base_complex_params)

    x0_c = x0_complex.copy()
    xhat0_c = xhat0_complex.copy()
    x0_c[1] *= vel_scale
    x0_c[3] *= vel_scale
    xhat0_c[1] *= vel_scale
    xhat0_c[3] *= vel_scale

    x_ref_complex = np.array([target_x, 0.0, 0.0, 0.0, 0.0, 0.0])
    K_tuned = lqr_gain(A, B, Q * ctrl_aggr, R)
    L_tuned = observer_gain_place(A, C, observer_poles * obs_speed)

    complex_result = run_closed_loop(
        x0=x0_c, xhat0=xhat0_c, x_ref=x_ref_complex,
        A=A, B=B, C=C, K=K_tuned, L=L_tuned, u_eq=u_eq,
        params=complex_params, mode="full"
    )

    plot_core_results(complex_result, "Section 4: Complex Landing Stress Test (Interactive)")
    plot_landing_scene(complex_result, complex_params, "Landing Geometry and Zones")

    final_landed, final_success, final_vel = check_landing(
        complex_result["X"][-1], complex_params["dims"],
        complex_params["safe_zone"], complex_params["hazard_zone"], complex_params["vel_threshold"]
    )

    section_metrics["sec4"] = {
        "landed": bool(final_landed),
        "safe": bool(final_success),
        "final_vel": [float(final_vel[0]), float(final_vel[1])],
        "touched_down": bool(complex_result["landed"]),
    }

    print("Landed:", bool(final_landed))
    print("Safe landing:", bool(final_success))
    print("Final velocity [dx, dy] m/s:", final_vel)
    print("Touched down during simulation:", bool(complex_result["landed"]))

sec4_target_x = widgets.FloatSlider(value=1.10, min=0.75, max=1.55, step=0.02, description="Target x")
sec4_vel = widgets.FloatSlider(value=1.0, min=0.7, max=1.6, step=0.05, description="Diag vel x")
sec4_ctrl = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description="Ctrl aggr")
sec4_obs = widgets.FloatSlider(value=1.0, min=0.6, max=1.8, step=0.05, description="Obs speed")

sec4_ui = widgets.VBox([sec4_target_x, sec4_vel, sec4_ctrl, sec4_obs])
sec4_out = widgets.interactive_output(run_section4, {
    "target_x": sec4_target_x,
    "vel_scale": sec4_vel,
    "ctrl_aggr": sec4_ctrl,
    "obs_speed": sec4_obs,
})
display(sec4_ui, sec4_out)

Output()

## Final Recap

> Quick pass/fail summary under default tuning values.

In [29]:
if not section_metrics:
    print("No interactive section has been run yet.")
    print("Move any slider in Sections 1-4 to populate this recap.")
else:
    print("Latest interactive recap:")
    for key in ["sec1", "sec2", "sec3", "sec4"]:
        if key in section_metrics:
            print(f"\n{key} -> {section_metrics[key]}")

Latest interactive recap:

sec1 -> {'initial_err': 1.2175385004179542, 'final_err': 0.018584947501570637, 'converged': True}

sec2 -> {'final_state_err': 0.9559595323441531, 'peak_thrust': 3.52856148765116, 'peak_phi': 0.35, 'stable_like': False}

sec3 -> {'final_tracking_err': 1.0499370123949638, 'final_estimation_err': 0.028854271525916613, 'stable_like': False}

sec4 -> {'landed': True, 'safe': False, 'final_vel': [1.6446277705919765, -2.7694692052145307], 'touched_down': True}
